In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("okienka").getOrCreate()
spark

In [5]:
czujnik_temperatury = ((12.5, "2019-01-02 12:00:00"),
(17.6, "2019-01-02 12:00:20"),
(14.6,  "2019-01-02 12:00:30"),
(22.9,  "2019-01-02 12:01:15"),
(17.4,  "2019-01-02 12:01:30"),
(25.8,  "2019-01-02 12:03:25"),
(27.1,  "2019-01-02 12:02:40"),
)

In [6]:
from pyspark.sql.functions import to_timestamp
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

schema = StructType([
    StructField("temperatura", DoubleType(), True),
    StructField("czas", StringType(), True),
])

SCHEMA = """
temperatura DOUBLE, czas STRING
""" # struktura DDL

In [7]:
df = (spark.createDataFrame(czujnik_temperatury, schema=schema)
      .withColumn("czas", to_timestamp("czas")))

df.printSchema()

root
 |-- temperatura: double (nullable = true)
 |-- czas: timestamp (nullable = true)



In [8]:
df.show(3)

+-----------+-------------------+
|temperatura|               czas|
+-----------+-------------------+
|       12.5|2019-01-02 12:00:00|
|       17.6|2019-01-02 12:00:20|
|       14.6|2019-01-02 12:00:30|
+-----------+-------------------+
only showing top 3 rows



In [9]:
df.createOrReplaceTempView("df") # widok tymczasowy - zeby moc uzywac sql

In [10]:
spark.sql("SELECT czas, temperatura FROM df WHERE temperatura > 21").show(5)

+-------------------+-----------+
|               czas|temperatura|
+-------------------+-----------+
|2019-01-02 12:01:15|       22.9|
|2019-01-02 12:03:25|       25.8|
|2019-01-02 12:02:40|       27.1|
+-------------------+-----------+



In [11]:
# Thumbling window

import pyspark.sql.functions as F

df2 = df.groupBy(F.window("czas","30 seconds")).count()
df2.show(truncate=False)

+------------------------------------------+-----+
|window                                    |count|
+------------------------------------------+-----+
|{2019-01-02 12:00:00, 2019-01-02 12:00:30}|2    |
|{2019-01-02 12:00:30, 2019-01-02 12:01:00}|1    |
|{2019-01-02 12:01:00, 2019-01-02 12:01:30}|1    |
|{2019-01-02 12:01:30, 2019-01-02 12:02:00}|1    |
|{2019-01-02 12:03:00, 2019-01-02 12:03:30}|1    |
|{2019-01-02 12:02:30, 2019-01-02 12:03:00}|1    |
+------------------------------------------+-----+



In [12]:
df2.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- count: long (nullable = false)



In [ ]:
# SPARK STRUCTURED STREAMING

In [13]:
df = spark.readStream.format("rate").option("rowsPerSecond", 1).load()

In [14]:
%%file streamrate.py
## uruchom przez spark-submit streamrate.py

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)


query = (df.writeStream 
    .format("console") 
    .outputMode("append") 
    .option("truncate", False) 
    .start()
) 

query.awaitTermination()

Overwriting streamrate.py


In [ ]:
# spark-submit

In [15]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()


df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

query = (df.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+---------+-----+
|timestamp|value|
+---------+-----+
+---------+-----+

Batch ID: 1
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-27 18:13:28.164|0    |
+-----------------------+-----+

Batch ID: 2
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-27 18:13:29.164|1    |
+-----------------------+-----+

Batch ID: 3
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-27 18:13:30.164|2    |
+-----------------------+-----+

Batch ID: 4
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-27 18:13:31.164|3    |
+-----------------------+-----+

Batch ID: 5
+-----------------------+-----+
|timestamp              |value|
+-----------------------+-----+
|2026-04-27 18:13:32.164|4    |
+-----------------------+-----+

Batch ID: 0
+----+-----------+
|czas|temperatura|


In [16]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura")>26 )

query = (stream.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [17]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()

from pyspark.sql.functions import col, expr

df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_filtered = stream.filter(col("temperatura") > 28)

query = (stream_filtered.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
#2. Prosty model wykrywający anomalie (bezstanowy)

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, expr, when

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=10):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )

stream_anomaly = stream.withColumn(
    "anomaly",
    when(col("temperatura") > 29.5, "TAK").otherwise("NIE")
)

query = (stream_anomaly.writeStream 
    .format("console") 
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [ ]:
#3. Okno czasowe: Tumbling Window

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, expr

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))

tumbling_window = (stream
    .groupBy(window(col("czas"), "10 seconds"))
    .avg("temperatura")
)

query = (tumbling_window.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

Batch ID: 0
+------+----------------+
|window|avg(temperatura)|
+------+----------------+
+------+----------------+

Batch ID: 1
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-21 12:36:00, 2026-04-21 12:36:10}|21.861183629303046|
|{2026-04-21 12:35:50, 2026-04-21 12:36:00}|23.771819142894262|
+------------------------------------------+------------------+

Batch ID: 2
+------------------------------------------+------------------+
|window                                    |avg(temperatura)  |
+------------------------------------------+------------------+
|{2026-04-21 12:36:00, 2026-04-21 12:36:10}|25.16351586264626 |
|{2026-04-21 12:35:50, 2026-04-21 12:36:00}|23.771819142894262|
+------------------------------------------+------------------+

Batch ID: 3
+------------------------------------------+------------------+
|window      

In [ ]:
# 4. Okno czasowe: Sliding Window

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window

spark = SparkSession.builder.appName("StreamingDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

def process_batch(df, batch_id, tstop=30):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()



df = (spark.readStream
      .format("rate")
      .option("rowsPerSecond", 1)
      .load()
)

stream = (df.withColumn("czas", col("timestamp"))
        .withColumn("temperatura", expr("20 + rand() * 10"))
        .select("czas", "temperatura")
       )
# Konwertuj 'temperatura' na typ Double, aby agregacja była możliwa
stream = stream.withColumn("temperatura", col("temperatura").cast("double"))


sliding = (stream.groupBy(window(col("czas"), "10 seconds", "5 seconds"))
           .avg("temperatura")
          )

query = (sliding.writeStream 
    .format("console") 
    .outputMode("complete")
    .foreachBatch(process_batch)
    .option("truncate", False) 
    .start()
)

In [20]:
%%file generator.py
# generator.py
import json, os, random, time
from datetime import datetime, timedelta

output_dir = "data/stream"
os.makedirs(output_dir, exist_ok=True)

sklepy = ['Warszawa', 'Kraków', 'Gdańsk', 'Wrocław']
kategorie = ['elektronika', 'odzież', 'żywność', 'książki']

def generate_transaction():
    return {
        'tx_id': f'TX{random.randint(1000,9999)}',
        'user_id': f'u{random.randint(1,20):02d}',
        'amount': round(random.uniform(5.0, 5000.0), 2),
        'store': random.choice(sklepy),
        'category': random.choice(kategorie),
        'timestamp': datetime.now().isoformat(),
    }

# Simulate file-based streaming
while True:
    batch = [generate_transaction() for _ in range(2)]
    filename = f"{output_dir}/events_{int(time.time())}.json"
    with open(filename, "w") as f:
        for e in batch:
            f.write(json.dumps(e) + "\n")
    print(f"Wrote: {filename}")
    time.sleep(5)

Overwriting generator.py


In [22]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, to_timestamp
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("jsonDemo").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

batch_counter = {"count": 0}

def process_batch(df, batch_id, tstop=5):
    batch_counter["count"] += 1
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)
    if batch_id == tstop:
        df.stop()
    


stream = (spark.readStream
          .schema(tx_schema)
          .json("data/stream"))

#### cos wyfiltrowac / pogrupowac (miasto, kat, suma per kategoria) outputmode zmienic jesli grupowanie
from pyspark.sql.functions import col, sum as _sum

# filtr
filtered = stream.filter(col("amount") > 50)

# agregacja
aggregated = (filtered
              .groupBy("store", "category")
              .agg(_sum("amount").alias("total_amount")))

query = (aggregated.writeStream
         .outputMode("complete")
         .format("console")
         .start())

In [23]:
a = (spark.read.schema(tx_schema).json("data/stream"))

In [24]:
a.show()

+------+-------+-------+--------+-----------+--------------------+
| tx_id|user_id| amount|   store|   category|           timestamp|
+------+-------+-------+--------+-----------+--------------------+
|TX9961|    u18|2734.57| Wrocław|    żywność|2026-04-21T12:42:...|
|TX9585|    u09| 392.21|Warszawa|    żywność|2026-04-21T12:42:...|
|TX5247|    u02| 2336.3|  Kraków|    książki|2026-04-21T12:51:...|
|TX4760|    u09|3260.35|  Kraków|    żywność|2026-04-21T12:51:...|
|TX3013|    u04|3094.72|  Kraków|    żywność|2026-04-21T12:51:...|
|TX6165|    u10|3837.23| Wrocław|     odzież|2026-04-21T12:51:...|
|TX4088|    u17|1007.24| Wrocław|elektronika|2026-04-21T12:51:...|
|TX6002|    u14|1229.02|  Kraków|    żywność|2026-04-21T12:51:...|
|TX3954|    u02| 639.31| Wrocław|    żywność|2026-04-21T12:42:...|
|TX2025|    u04|4925.21| Wrocław|elektronika|2026-04-21T12:42:...|
|TX4428|    u20|2136.77|  Kraków|    książki|2026-04-21T12:42:...|
|TX2741|    u17|1536.11|Warszawa|     odzież|2026-04-21T12:42: